In [ ]:
#!/usr/bin/env python3
"""
etl_transform.py
- Reads raw/raw.csv
- Cleans and filters rows
- Derives trip_duration_min and cost_per_mile
- Writes processed outputs (cleaned + aggregates + quality report)
"""

# Install necessary libraries (no requirements.txt)
import sys
import subprocess
import importlib

def _ensure(import_name: str, pip_name: str) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pip_name])

_ensure("pandas", "pandas")
_ensure("numpy", "numpy")

# Import libraries
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd

def main() -> None:
    in_path = Path(os.getenv("TAXI_RAW_IN", "raw/raw.csv"))
    out_dir = Path(os.getenv("TAXI_PROCESSED_DIR", "processed"))
    out_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(in_path)
    raw_rows = len(df)

    required = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "total_amount"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns: {missing}")

    df["pickup_dt"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    df["dropoff_dt"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

    df["trip_duration_min"] = (df["dropoff_dt"] - df["pickup_dt"]).dt.total_seconds() / 60.0

    for c in ["trip_distance", "total_amount", "fare_amount", "tip_amount", "tolls_amount", "passenger_count"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["cost_per_mile"] = np.where(
        df["trip_distance"].fillna(0) > 0,
        df["total_amount"] / df["trip_distance"],
        np.nan,
    )

    # Default filters: tuned for a demo subset; override via env vars if needed
    min_dur = float(os.getenv("TAXI_MIN_DURATION_MIN", "1.0"))
    max_dur = float(os.getenv("TAXI_MAX_DURATION_MIN", "180.0"))
    min_dist = float(os.getenv("TAXI_MIN_DISTANCE_MI", "0.1"))
    max_dist = float(os.getenv("TAXI_MAX_DISTANCE_MI", "50.0"))
    min_total = float(os.getenv("TAXI_MIN_TOTAL_AMOUNT", "2.0"))
    max_total = float(os.getenv("TAXI_MAX_TOTAL_AMOUNT", "500.0"))
    max_pax = int(os.getenv("TAXI_MAX_PASSENGERS", "8"))

    filters = {}
    filters["valid_datetimes"] = df["pickup_dt"].notna() & df["dropoff_dt"].notna()
    filters["duration_range"] = df["trip_duration_min"].between(min_dur, max_dur, inclusive="both")
    filters["distance_range"] = df["trip_distance"].between(min_dist, max_dist, inclusive="both")
    filters["total_amount_range"] = df["total_amount"].between(min_total, max_total, inclusive="both")

    if "passenger_count" in df.columns:
        filters["passenger_range"] = df["passenger_count"].between(1, max_pax, inclusive="both")
    else:
        filters["passenger_range"] = pd.Series([True] * len(df), index=df.index)

    mask = filters["valid_datetimes"] & filters["duration_range"] & filters["distance_range"] & filters["total_amount_range"] & filters["passenger_range"]
    clean = df.loc[mask].copy()
    clean_rows = len(clean)

    clean["pickup_hour"] = clean["pickup_dt"].dt.hour.astype("Int64")

    agg_by_hour = (
        clean.groupby("pickup_hour", dropna=True)
        .agg(
            trips=("pickup_hour", "size"),
            total_revenue=("total_amount", "sum"),
            avg_distance=("trip_distance", "mean"),
            avg_duration_min=("trip_duration_min", "mean"),
            avg_cost_per_mile=("cost_per_mile", "mean"),
        )
        .reset_index()
        .sort_values("pickup_hour")
    )

    clean_path = out_dir / "trips_clean.csv"
    agg_path = out_dir / "aggregates_by_hour.csv"
    report_path = out_dir / "data_quality_report.json"

    clean.to_csv(clean_path, index=False)
    agg_by_hour.to_csv(agg_path, index=False)

    report = {
        "raw_rows": int(raw_rows),
        "clean_rows": int(clean_rows),
        "dropped_rows": int(raw_rows - clean_rows),
        "drop_rate": float((raw_rows - clean_rows) / raw_rows) if raw_rows else 0.0,
        "filter_fail_counts": {k: int((~v).sum()) for k, v in filters.items()},
        "derived_columns": ["pickup_dt", "dropoff_dt", "trip_duration_min", "cost_per_mile", "pickup_hour"],
    }
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

    print(f"Raw rows: {raw_rows}", flush=True)
    print(f"Clean rows: {clean_rows}", flush=True)
    print(f"Wrote {clean_path}", flush=True)
    print(f"Wrote {agg_path}", flush=True)
    print(f"Wrote {report_path}", flush=True)

if __name__ == "__main__":
    main()
